In [ ]:
# to read faa data

import polars as pl

strikes = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class10/refs/heads/main/data/faa_strikes.txt",
    separator="\t"
)

strikes = strikes.with_columns(
    pl.col("Collision Date and Time").str.strptime(pl.Datetime).alias("Collision Date and Time")
)

print(strikes)

shape: (28_298, 25)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Airport:  ┆ Airport:  ┆ Origin    ┆ Origin    ┆ … ┆ Wildlife: ┆ Wildlife: ┆ Number of ┆ Record   │
│ Code      ┆ Name      ┆ State     ┆ State     ┆   ┆ Species   ┆ Species   ┆ Strikes   ┆ ID       │
│ ---       ┆ ---       ┆ ---       ┆ Code      ┆   ┆ ---       ┆ ID        ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ ---       ┆   ┆ str       ┆ ---       ┆ i64       ┆ i64      │
│           ┆           ┆           ┆ str       ┆   ┆           ┆ str       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ KAAF      ┆ APALACHIC ┆ Florida   ┆ FL        ┆ … ┆ Domestic  ┆ 1F12      ┆ 1         ┆ 17459    │
│           ┆ OLA       ┆           ┆           ┆   ┆ dog       ┆           ┆           ┆          │
│           ┆ REGIONAL  ┆           ┆           ┆   ┆           ┆      

In [4]:
# Question 1

import polars as pl

strikes = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class10/refs/heads/main/data/faa_strikes.txt",
    separator="\t"
)

strikes = strikes.with_columns(
    pl.col("Collision Date and Time").str.strptime(pl.Datetime)
)

running_total_strikes = (strikes
                         .with_columns(pl.col("Collision Date and Time").dt.date().alias("date"))
                         .group_by("date")
                         .agg(pl.col("Number of Strikes").sum().alias("daily_strikes"))
                         .sort("date")
                         .with_columns(
                             pl.col("daily_strikes")
                             .cum_sum()
                             .alias("strikes_cumulative")
                         )
                         .filter(pl.col("date") <= pl.date(2013, 12, 31))
                         )

running_total_strikes

date,daily_strikes,strikes_cumulative
date,i64,i64
2000-01-02,1,1
2000-01-03,2,3
2000-01-05,3,6
2000-01-06,1,7
2000-01-08,4,11
…,…,…
2013-12-27,6,24096
2013-12-28,1,24097
2013-12-29,2,24099


In [9]:
# Question 2

damage_state = (
    strikes
    .group_by("Origin State")
    .agg(pl.col("Cost: Total $").sum().alias("damage"))
    .with_columns(
        pl.col("damage")
        .rank("dense", descending=True)
        .alias("ranking")
    )
    .filter(pl.col("ranking") == 3)
    )
damage_state

Origin State,damage,ranking
str,i64,u32
"""California""",29671432,3


In [11]:
# Question 3

type_species = (strikes.group_by(["Aircraft: Type", "Wildlife: Species Group"])
  .agg(pl.col("Cost: Total $").sum().alias("damage"))
  .with_columns(
      pl.col("damage")
      .rank("dense")
      .over("Aircraft: Type")
      .alias("ranking")
  )
  .filter(pl.col("ranking") == 2)
  )
type_species

Aircraft: Type,Wildlife: Species Group,damage,ranking
str,str,i64,u32
"""Helicopter""","""Doves""",75,2
"""Airplane""","""Wolves, Dogs, Foxes""",250,2
"""NA""","""Caracaras, Falcons""",1667,2


In [13]:
# Question 4

greatest_strike_increase = (strikes.with_columns(pl.col("Collision Date and Time").dt.date().alias("date"))
  .group_by("date")
  .agg(pl.col("Number of Strikes").sum().alias("daily_strikes"))
  .sort("date")
  .with_columns(
      (pl.col("daily_strikes") - pl.col("daily_strikes").shift(1)).alias("delta_strikes")
  )
  .sort("delta_strikes", descending=True)
)
greatest_strike_increase

date,daily_strikes,delta_strikes
date,i64,i64
2000-01-02,1,null
2010-10-29,25,18
2014-08-25,25,17
2012-06-28,21,16
2009-07-07,21,14
…,…,…
2012-06-29,6,-15
2012-10-14,5,-15
2014-08-26,10,-15


In [15]:
# Question 5

greatest_strike_increase = (
    strikes
    .with_columns(pl.col("Collision Date and Time").dt.date().alias("date"))
    .group_by(["Aircraft: Type", "date"])
    .agg(pl.col("Number of Strikes").sum().alias("daily_strikes"))
    .sort(["Aircraft: Type", "date"])
    .with_columns(
        (pl.col("daily_strikes") - pl.col("daily_strikes").shift(1).over("Aircraft: Type")).alias("delta_strikes")
    )
    .filter(pl.col("delta_strikes") > 0)
    .sort(["Aircraft: Type", "delta_strikes", "date"], descending=[False, True, False])
    .with_columns(
        pl.col("delta_strikes")
        .rank(method="ordinal", descending=True)
        .over("Aircraft: Type")
        .alias("ranking")
    )
    .filter(pl.col("ranking") == 1)
    .select(["Aircraft: Type", "date", "daily_strikes", "delta_strikes", "ranking"])
)

greatest_strike_increase

Aircraft: Type,date,daily_strikes,delta_strikes,ranking
str,date,i64,i64,u32
"""Airplane""",2014-08-25,25,17,1
"""Helicopter""",2010-07-21,2,1,1
"""NA""",2014-08-04,5,4,1
